# Part 2 ->resume on Kaggle GPU

This notebook is the **resume version** of the part-2 run. Most of the work is already done and is skipped, not re-run.

In [ ]:
# CELL 1: check GPU + internet and install the one extra package this project needs
import subprocess, sys, torch, urllib.request
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: set Session options > Accelerator > GPU')
try:
    urllib.request.urlopen('https://huggingface.co', timeout=8); print('Internet: ON')
except Exception as e:
    print('Internet: OFF. Turn it on in Session options, then restart the session.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
import segmentation_models_pytorch, transformers
print('segmentation_models_pytorch', segmentation_models_pytorch.__version__, '| transformers', transformers.__version__)
import os
os.makedirs('/kaggle/working/code', exist_ok=True)
print('code folder ready:', os.path.isdir('/kaggle/working/code'))


In [ ]:
# CELL 1b: create the folder the code files are written into
import os
os.makedirs('/kaggle/working/code', exist_ok=True)
print('folder ready:', os.path.isdir('/kaggle/working/code'))

In [ ]:
%%writefile /kaggle/working/code/config.py
"""Central configuration for the whole project.

Every notebook imports this file, so all experiments share the same
classes, splits, sizes and hyperparameters (the brief requires an
identical pipeline for both models).
"""

import os
import json
import random
import numpy as np

# paths
WORK_DIR     = '/kaggle/working'
LOCAL_DS     = '/tmp/ds'                                     # fast local copy of images + masks
PROJECT_DIR  = os.path.join(WORK_DIR, 'project')             # everything saved here is kept as notebook output
CODE_DIR     = os.path.join(WORK_DIR, 'code')
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'results')          # every metric lands here
FIGURES_DIR  = os.path.join(RESULTS_DIR, 'figures')
CKPT_DIR     = os.path.join(PROJECT_DIR, 'checkpoints')      # resume-safe checkpoints
MASKS_INDEX  = os.path.join(LOCAL_DS, 'masks_index.csv')     # photo -> mask mapping (absolute paths)
MASKS_DIR    = os.path.join(LOCAL_DS, 'masks')

#  data
CLASSES = ['person', 'car', 'dog', 'cat', 'chair', 'bottle',
           'bus', 'bicycle', 'horse', 'cup', 'laptop', 'bird']
NUM_CLASSES = 1 + len(CLASSES)          # 0 = background, 1..12 = classes above
CLASS_NAMES = ['background'] + CLASSES
IMG_SIZE = 512                          # all images resized to IMG_SIZE x IMG_SIZE
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# training
# Identical pipeline for both models 
OPTIMIZER     = 'adamw'
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
LR_SCHEDULE   = 'poly'      # poly decay, power below; standard for segmentation
POLY_POWER    = 0.9
EPOCHS_MAIN   = 12          # full training runs
EPOCHS_SWEEP  = 6           # loss-comparison runs (same for every loss = fair)
EPOCHS_TRIAL  = 2           # each Optuna trial (cheap proxy; see agent_search docstring)
BATCH_SIZE    = {'deeplabv3plus': 8, 'segformer_b3': 4}
GRAD_ACCUM    = {'deeplabv3plus': 1, 'segformer_b3': 2}   # effective batch = BATCH*ACCUM
NUM_WORKERS   = 4
USE_AMP       = True        # mixed precision on GPU (T4-safe)

# agent
AGENT_N_TRIALS    = 10        # compute budget for the composite-weight agent
AGENT_TRAIN_SUBSET = 1500     # images per trial (documented proxy, see report text)
THRESHOLD_GRID    = [round(0.05 * i, 2) for i in range(0, 14)]   # 0.00 .. 0.65
AGENT_SEED        = 42

# seeds / final
SEEDS = [42, 1337, 2026]    # >= 3 seeds for mean +/- std (brief requirement)

MODELS = ['deeplabv3plus', 'segformer_b3']
LOSSES = ['ce', 'weighted_ce', 'focal', 'dice', 'tversky', 'lovasz']


def set_seed(seed):
    """Seed every RNG we use so runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def ensure_dirs():
    for d in (PROJECT_DIR, RESULTS_DIR, FIGURES_DIR, CKPT_DIR):
        os.makedirs(d, exist_ok=True)


def as_dict():
    """Full hyperparameter table, dumped into results/config.json (reproducibility)."""
    keys = [k for k in dir() if k.isupper()]
    return {k: globals()[k] for k in keys}


def save_config():
    ensure_dirs()
    with open(os.path.join(RESULTS_DIR, 'config.json'), 'w') as f:
        json.dump(as_dict(), f, indent=2, default=str)

In [ ]:
%%writefile /kaggle/working/code/metrics.py
"""Evaluation metrics - pure NumPy so they can be tested without a GPU.

(Part D):
per-class IoU, mIoU, Dice/F1, Precision, Recall, Pixel Accuracy,
plus a boundary metric (Boundary F1 with a pixel tolerance) and a
pixel-level confusion matrix for the error analysis.
"""

import numpy as np
from scipy import ndimage


def confusion_matrix(pred, target, num_classes):
    """Pixel-level confusion matrix. Rows = true class, columns = predicted."""
    pred = np.asarray(pred).ravel().astype(np.int64)
    target = np.asarray(target).ravel().astype(np.int64)
    valid = (target >= 0) & (target < num_classes) & (pred >= 0) & (pred < num_classes)
    cm = np.bincount(num_classes * target[valid] + pred[valid],
                     minlength=num_classes ** 2)
    return cm.reshape(num_classes, num_classes)


def metrics_from_confusion(cm):
    """All Part-D metrics from one confusion matrix."""
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    def safe_div(a, b):
        out = np.full_like(a, np.nan, dtype=np.float64)
        np.divide(a, b, out=out, where=(b > 0))
        return out

    iou = safe_div(tp, tp + fp + fn)
    dice = safe_div(2 * tp, 2 * tp + fp + fn)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    pixel_acc = tp.sum() / cm.sum() if cm.sum() > 0 else float('nan')

    present = (tp + fn) > 0          # classes that actually occur in the ground truth
    return {
        'per_class_iou': iou.tolist(),
        'per_class_dice': dice.tolist(),
        'per_class_precision': precision.tolist(),
        'per_class_recall': recall.tolist(),
        'miou': float(np.nanmean(iou[present])) if present.any() else float('nan'),
        'mean_dice': float(np.nanmean(dice[present])) if present.any() else float('nan'),
        'mean_precision': float(np.nanmean(precision[present])) if present.any() else float('nan'),
        'mean_recall': float(np.nanmean(recall[present])) if present.any() else float('nan'),
        'pixel_accuracy': float(pixel_acc),
    }


def _boundary(binary_mask):
    """1-pixel inner boundary of a binary mask."""
    eroded = ndimage.binary_erosion(binary_mask, border_value=0)
    return binary_mask & ~eroded


def boundary_f1(pred, target, num_classes, tolerance=2):
    """Boundary F1: a predicted boundary pixel counts as correct if it lies
    within `tolerance` pixels of a ground-truth boundary pixel (and vice
    versa). Averaged over classes present in target or prediction."""
    scores = []
    for c in range(num_classes):
        gt = target == c
        pr = pred == c
        if not gt.any() and not pr.any():
            continue
        gt_b, pr_b = _boundary(gt), _boundary(pr)
        if not gt_b.any() and not pr_b.any():
            continue
        if not gt_b.any() or not pr_b.any():
            scores.append(0.0)
            continue
        dist_gt = ndimage.distance_transform_edt(~gt_b)
        dist_pr = ndimage.distance_transform_edt(~pr_b)
        tp_p = (dist_gt[pr_b] <= tolerance).sum()   # pred boundary close to gt
        tp_r = (dist_pr[gt_b] <= tolerance).sum()   # gt boundary close to pred
        prec = tp_p / pr_b.sum()
        rec = tp_r / gt_b.sum()
        f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
        scores.append(f1)
    return float(np.mean(scores)) if scores else float('nan')


def full_report(pred, target, num_classes, class_names, tolerance=2):
    """One image pair -> complete metric dictionary."""
    cm = confusion_matrix(pred, target, num_classes)
    m = metrics_from_confusion(cm)
    m['boundary_f1'] = boundary_f1(pred, target, num_classes, tolerance)
    m['class_names'] = list(class_names)
    return m


def mean_std(dicts, keys):
    """Mean +/- std over seed runs, as {'key': (mean, std)}."""
    out = {}
    for k in keys:
        vals = [d[k] for d in dicts if d.get(k) is not None and not np.isnan(d.get(k, np.nan))]
        if vals:
            out[k] = (float(np.mean(vals)), float(np.std(vals)))
    return out

In [ ]:
%%writefile /kaggle/working/code/losses.py
"""Loss functions (Part C).

Six individual losses from different families, plus a composite loss
L_total = sum_k lambda_k * L_k  whose weights the agent tunes.

  ce          Cross-entropy            (distribution-based; the default)
  weighted_ce CE with median-frequency class weights (distribution; imbalance)
  focal       Focal loss               (distribution; focuses hard pixels)
  dice        Soft Dice loss           (region-based; matches Dice metric)
  tversky     Tversky loss             (region-based; FP/FN trade-off knob)
  lovasz      Lovasz-Softmax           (metric-based; directly optimises IoU)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE_INDEX = -1


# ------------------------------------------------------------- helpers
def _one_hot(target, num_classes, ignore_index=IGNORE_INDEX):
    """(N,H,W) int -> (N,C,H,W) float one-hot, ignoring ignore_index."""
    valid = target != ignore_index
    t = target.clone()
    t[~valid] = 0
    oh = F.one_hot(t, num_classes).permute(0, 3, 1, 2).float()
    return oh * valid.unsqueeze(1).float(), valid


class CrossEntropy(nn.Module):
    def __init__(self, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, logits, target):
        return self.ce(logits, target)


class WeightedCrossEntropy(nn.Module):
    """CE weighted per class, e.g. median-frequency weights from the
    class-pixel distribution (a data-side answer to imbalance)."""
    def __init__(self, class_weights, ignore_index=IGNORE_INDEX):
        super().__init__()
        w = torch.as_tensor(class_weights, dtype=torch.float32)
        self.register_buffer('w', w)
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        return F.cross_entropy(logits, target, weight=self.w,
                               ignore_index=self.ignore_index)


class FocalLoss(nn.Module):
    """Down-weights easy pixels so the model focuses on hard ones
    (Lin et al., 2017). Good for dominant-background imbalance."""
    def __init__(self, gamma=2.0, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none',
                             ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        valid = target != self.ignore_index
        loss = ((1 - pt) ** self.gamma) * ce
        return loss[valid].mean()


class DiceLoss(nn.Module):
    """Soft Dice: 1 - mean_c (2|p_c ∩ y_c| + eps) / (|p_c| + |y_c| + eps).
    Directly optimises a soft version of the Dice/F1 metric."""
    def __init__(self, smooth=1.0, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        C = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        oh, valid = _one_hot(target, C, self.ignore_index)
        dims = (0, 2, 3)
        inter = (probs * oh).sum(dims)
        union = probs.sum(dims) + oh.sum(dims)
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class TverskyLoss(nn.Module):
    """Generalises Dice with separate FP/FN weights. beta > alpha
    penalises false negatives more (recall-leaning)."""
    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        C = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        oh, valid = _one_hot(target, C, self.ignore_index)
        dims = (0, 2, 3)
        tp = (probs * oh).sum(dims)
        fp = (probs * (1 - oh)).sum(dims)
        fn = ((1 - probs) * oh).sum(dims)
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1 - tv.mean()


# ------------------------------------------------------------- Lovasz
def _lovasz_grad(gt_sorted):
    """Gradient of the Lovasz extension of the IoU w.r.t. sorted errors."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def _lovasz_softmax_flat(probs, labels):
    """probs (P,C) softmaxed, labels (P,) int."""
    C = probs.shape[1]
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg[perm])))
    if not losses:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()


class LovaszSoftmax(nn.Module):
    """Lovasz-Softmax (Berman et al., 2018): a convex surrogate that
    optimises the IoU (Jaccard) metric directly."""
    def __init__(self, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        probs = F.softmax(logits, dim=1)
        probs = probs.permute(0, 2, 3, 1).reshape(-1, probs.shape[1])
        labels = target.reshape(-1)
        valid = labels != self.ignore_index
        return _lovasz_softmax_flat(probs[valid], labels[valid])


class CompositeLoss(nn.Module):
    """L_total = sum_k lambda_k * L_k. The agent (code/agent_search.py)
    chooses the lambdas on the validation set, never on test."""
    def __init__(self, components, weights):
        super().__init__()
        assert set(components) == set(weights)
        self.names = list(components)
        self.components = nn.ModuleDict(components)
        self.weights = {k: float(v) for k, v in weights.items()}

    def forward(self, logits, target):
        total = 0.0
        for k in self.names:
            total = total + self.weights[k] * self.components[k](logits, target)
        return total

    def describe(self):
        return {k: self.weights[k] for k in self.names}


def median_frequency_weights(counts, num_classes):
    """class weight = median(freq) / freq, freq = pixel share of the class.
    Rare classes get large weights. Background is included."""
    counts = np.asarray(counts, dtype=np.float64) + 1.0
    freq = counts / counts.sum()
    w = np.median(freq) / freq
    return (w / w.mean()).tolist()          # normalise so mean weight = 1


import numpy as np  # noqa: E402  (kept low so the torch API above reads first)

LOSS_REGISTRY = {
    'ce':          lambda C, w=None: CrossEntropy(),
    'weighted_ce': lambda C, w=None: WeightedCrossEntropy(w),
    'focal':       lambda C, w=None: FocalLoss(gamma=2.0),
    'dice':        lambda C, w=None: DiceLoss(),
    'tversky':     lambda C, w=None: TverskyLoss(alpha=0.3, beta=0.7),
    'lovasz':      lambda C, w=None: LovaszSoftmax(),
}


def build_loss(name, num_classes, class_weights=None):
    return LOSS_REGISTRY[name](num_classes, class_weights)

In [ ]:
%%writefile /kaggle/working/code/models.py
"""The two models (Part B-1).

  deeplabv3plus  - CNN: ResNet-50 encoder + atrous spatial pyramid pooling
                   + decoder (Chen et al., 2018), via segmentation_models_pytorch
  segformer_b3   - Transformer: MiT-B3 hierarchical encoder + all-MLP decoder
                   (Xie et al., 2021), via HuggingFace transformers

Both are wrapped so forward() returns an logits tensor of shape
(N, num_classes, H, W) - identical interface, identical pipeline.
"""

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import config as C


class SegformerWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation
        self.net = SegformerForSemanticSegmentation.from_pretrained(
            'nvidia/mit-b3', num_labels=num_classes, ignore_mismatched_sizes=True)

    def forward(self, x):
        out = self.net(pixel_values=x).logits          # (N, C, H/4, W/4)
        return F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)


class DeepLabWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.net = smp.DeepLabV3Plus(encoder_name='resnet50', encoder_weights='imagenet',
                                     in_channels=3, classes=num_classes)

    def forward(self, x):
        return self.net(x)


def build_model(name, num_classes=C.NUM_CLASSES):
    if name == 'deeplabv3plus':
        return DeepLabWrapper(num_classes)
    if name == 'segformer_b3':
        return SegformerWrapper(num_classes)
    raise ValueError(f"unknown model {name}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def measure_flops(model, img_size=C.IMG_SIZE, device='cpu'):
    """GFLOPs for one forward pass at the training input size (uses thop)."""
    try:
        from thop import profile
        m = model.to(device).eval()
        x = torch.randn(1, 3, img_size, img_size, device=device)
        with torch.no_grad():
            flops, _ = profile(m, inputs=(x,), verbose=False)
        return flops / 1e9
    except Exception as e:                       # noqa: BLE001
        print(f"FLOP measurement failed ({e}); record as None and mention it.")
        return None


def measure_inference_time(model, img_size=C.IMG_SIZE, device='cuda', n=30):
    """Mean milliseconds per image, batch size 1, after warmup."""
    m = model.to(device).eval()
    x = torch.randn(1, 3, img_size, img_size, device=device)
    with torch.no_grad():
        for _ in range(5):
            m(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n):
            m(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) / n
    return dt * 1000.0


def model_card(name, device='cuda', img_size=C.IMG_SIZE):
    """The Part-B-1 reporting table for one model."""
    model = build_model(name)
    total, trainable = count_params(model)
    card = {
        'model': name,
        'input_size': img_size,
        'params_total': int(total),
        'params_trainable': int(trainable),
        'gflops': measure_flops(model, img_size, device),
        'inference_ms_per_image': measure_inference_time(model, img_size, device),
    }
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()
    return card

In [ ]:
%%writefile /kaggle/working/code/data.py
"""Dataset and loaders. One shared pipeline for BOTH models (fair comparison).

Reads the masks written by the masks walkthrough:
  masks_index.csv  - one row per image: which photo belongs to which mask PNG
  masks/{train,val,test}/*.png - single-channel class-id masks (0 = background)

The masks-index column names are auto-detected, because the exact header
depends on how the walkthrough saved it.
"""

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

import torch
from torch.utils.data import Dataset, DataLoader

import config as C

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # fix for the truncated image 


# ------------------------------------------------------- index resolution
def _find_col(cols, needles):
    for col in cols:
        low = col.lower()
        if any(n in low for n in needles):
            return col
    return None


def resolve_masks_index(index_path=None, verbose=True):
    """Return a DataFrame with columns [image, mask, split]."""
    index_path = index_path or C.MASKS_INDEX
    if not os.path.exists(index_path):
        raise FileNotFoundError(
            f"masks index not found at {index_path}. Did the masks "
            f"walkthrough (Cell 4) finish? It writes masks_index.csv.")
    df = pd.read_csv(index_path)
    img_col = _find_col(df.columns, ['image', 'filepath', 'photo', 'img'])
    msk_col = _find_col(df.columns, ['mask'])
    if img_col is None or msk_col is None or img_col == msk_col:
        raise ValueError(
            f"Could not auto-detect image/mask columns. Found: {list(df.columns)}.")
    split_col = _find_col(df.columns, ['split'])
    out = pd.DataFrame({'image': df[img_col].astype(str),
                        'mask': df[msk_col].astype(str)})
    if split_col is not None:
        out['split'] = (df[split_col].astype(str).str.lower()
                        .str.replace('validation', 'val').str.replace('val_split', 'val')
                        .str.replace('test_split', 'test'))
    else:
        # fall back to which masks/<split>/ folder the mask sits in
        def split_from_path(p):
            for s in ('train', 'val', 'test'):
                if f'/{s}/' in p.replace('\\', '/'):
                    return s
            return 'unknown'
        out['split'] = out['mask'].map(split_from_path)

    def fix(p, kind):
        p = p.replace('\\', '/')
        if os.path.exists(p):
            return p
        base = os.path.basename(p)
        roots = ([C.MASKS_DIR] if kind == 'mask'
                 else [])
        for root in roots:
            hits = glob.glob(os.path.join(root, '**', base), recursive=True)
            if hits:
                return hits[0]
        return p  # leave broken; dataset init reports it clearly

    out['image'] = out['image'].map(lambda p: fix(p, 'image'))
    out['mask'] = out['mask'].map(lambda p: fix(p, 'mask'))
    n_missing = int((~out['image'].map(os.path.exists)).sum()
                    + (~out['mask'].map(os.path.exists)).sum())
    if verbose:
        print(f"masks index: {len(out)} rows | splits: "
              f"{out['split'].value_counts().to_dict()} | missing files: {n_missing}")
        print(out.head(2).to_string())
    if n_missing:
        raise FileNotFoundError(f"{n_missing} files from the index are missing on Drive. ")
    return out


# ------------------------------------------------------- dataset
class CocoSegDataset(Dataset):
    """(image tensor 3xHxW float, mask HxW long) with identical resize for
    both models. Train mode applies the augmentation pipeline."""

    def __init__(self, frame, split, img_size=C.IMG_SIZE, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.split = split
        self.size = img_size
        self.augment = augment
        bad = [i for i in range(min(len(self.frame), 25))
               if not (os.path.exists(self.frame['image'][i]) and os.path.exists(self.frame['mask'][i]))]
        if bad:
            raise FileNotFoundError(f"Missing files, e.g. row {bad[0]}: "
                                    f"{self.frame['image'][bad[0]]}")

    def __len__(self):
        return len(self.frame)

    def _augment(self, img, mask, rng):
        # horizontal flip
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        # scale jitter 0.75 - 1.25, then center crop / pad back to size
        s = float(rng.uniform(0.75, 1.25))
        w = h = self.size
        nw, nh = max(8, int(round(w * s))), max(8, int(round(h * s)))
        img = img.resize((nw, nh), Image.BILINEAR)
        mask = mask.resize((nw, nh), Image.NEAREST)
        canvas_i = Image.new('RGB', (w, h))
        canvas_m = Image.new('L', (w, h))
        ox = (nw - w) // 2
        oy = (nh - h) // 2
        # crop if larger, paste if smaller
        src = (max(0, ox), max(0, oy), max(0, ox) + w, max(0, oy) + h)
        img = img.crop(src) if nw >= w and nh >= h else img
        mask = mask.crop(src) if nw >= w and nh >= h else mask
        if img.size != (w, h):
            img = img.resize((w, h), Image.BILINEAR)
            mask = mask.resize((w, h), Image.NEAREST)
        # colour jitter (image only, never the mask)
        img = ImageEnhance.Brightness(img).enhance(float(rng.uniform(0.8, 1.2)))
        img = ImageEnhance.Contrast(img).enhance(float(rng.uniform(0.8, 1.2)))
        img = ImageEnhance.Color(img).enhance(float(rng.uniform(0.8, 1.2)))
        return img, mask

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(row['image']).convert('RGB').resize(
            (self.size, self.size), Image.BILINEAR)
        mask = Image.open(row['mask']).resize(
            (self.size, self.size), Image.NEAREST)
        if self.augment:
            rng = np.random.default_rng(torch.initial_seed() % (2**31) + i)
            img, mask = self._augment(img, mask, rng)
        x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0)
        x = x.permute(2, 0, 1)
        mean = torch.tensor(C.IMAGENET_MEAN).view(3, 1, 1)
        std = torch.tensor(C.IMAGENET_STD).view(3, 1, 1)
        x = (x - mean) / std
        y = torch.from_numpy(np.asarray(mask, dtype=np.int64))
        return x, y


def get_loader(frame, split, batch_size, augment=False, subset=None,
               seed=42, shuffle=None, num_workers=C.NUM_WORKERS):
    """One loader factory used by every experiment."""
    if subset:
        frame = frame.sample(n=min(subset, len(frame)), random_state=seed)
    ds = CocoSegDataset(frame, split, augment=augment)
    if shuffle is None:
        shuffle = (split == 'train')
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True, drop_last=shuffle)


def load_frames():
    """Resolve the index once and return {'train','val','test'} DataFrames."""
    idx = resolve_masks_index()
    frames = {s: idx[idx.split == s] for s in ('train', 'val', 'test')}
    for s, f in frames.items():
        if len(f) == 0:
            raise ValueError(f"Split '{s}' is empty. Splits found: "
                             f"{sorted(idx.split.unique())}.")
    return frames

In [ ]:
%%writefile /kaggle/working/code/engine.py
"""Training and evaluation loops, shared by every experiment.

Checkpointing: after every epoch a checkpoint is written to Drive, and
train() resumes automatically if one exists - so a Colab disconnect
never loses more than one epoch.
"""

import os
import json
import time
import numpy as np
import torch
import torch.nn.functional as F

import config as C
import metrics as M


def _device():
    return 'cuda' if torch.cuda.is_available() else 'cpu'


def _poly_lr(epoch, max_epochs):
    return (1 - epoch / max(max_epochs, 1)) ** C.POLY_POWER


def ckpt_path(tag):
    return os.path.join(C.CKPT_DIR, f'{tag}_ckpt.pt')


def history_path(tag):
    return os.path.join(C.RESULTS_DIR, f'{tag}_history.json')


def train(model, tag, train_loader, val_loader, loss_fn, epochs,
          lr=C.LR, seed=42, verbose=True):
    """Train with checkpoint-resume. Returns the history list.

    tag uniquely names the run (model/loss/seed/improvement combo), so
    every experiment keeps its own checkpoint and history.
    """
    C.ensure_dirs()
    C.set_seed(seed)
    dev = _device()
    model = model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=C.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: _poly_lr(e, epochs))
    scaler = torch.amp.GradScaler('cuda', enabled=(C.USE_AMP and dev == 'cuda'))
    accum = C.GRAD_ACCUM.get(tag.split('_')[0], 1)

    start_epoch, history = 0, []
    path = ckpt_path(tag)
    if os.path.exists(path):
        state = torch.load(path, map_location=dev, weights_only=False)
        model.load_state_dict(state['model'])
        opt.load_state_dict(state['opt'])
        sched.load_state_dict(state['sched'])
        scaler.load_state_dict(state['scaler'])
        start_epoch = state['epoch'] + 1
        history = state['history']
        if start_epoch < epochs:
            print(f'[resume] {tag}: continuing at epoch {start_epoch + 1}/{epochs}')
        else:
            print(f'[resume] {tag}: already trained for {epochs} epochs, skipping')

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        model.train()
        running, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
                loss = loss_fn(model(x), y) / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0 or step + 1 == len(train_loader):
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            running += float(loss.detach()) * accum
            nb += 1
        sched.step()
        train_loss = running / max(nb, 1)
        val = evaluate(model, val_loader)
        secs = time.time() - t0
        history.append({'epoch': epoch + 1, 'train_loss': train_loss,
                        'val_miou': val['miou'], 'epoch_seconds': secs})
        if verbose:
            print(f'{tag} epoch {epoch + 1}/{epochs}  loss {train_loss:.4f}  '
                  f'val mIoU {val["miou"]:.4f}  ({secs:.0f}s)')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                    'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'history': history}, path)
        with open(history_path(tag), 'w') as f:
            json.dump(history, f, indent=2)
    return history


@torch.no_grad()
def evaluate(model, loader, threshold=0.0, tta=False, max_batches=None):
    """Full validation/test metric set (Part D formulas).

    threshold: pixels whose top softmax probability is below `threshold`
    are assigned to background. 0.0 = plain argmax (the default).
    tta: average softmax over the image and its horizontal flip.
    """
    dev = _device()
    model = model.to(dev).eval()
    cm = np.zeros((C.NUM_CLASSES, C.NUM_CLASSES), dtype=np.int64)
    bf1 = []
    for bi, (x, y) in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        x = x.to(dev)
        with torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
            probs = F.softmax(model(x).float(), dim=1)
            if tta:
                probs = probs + torch.flip(F.softmax(
                    model(torch.flip(x, dims=[3])).float(), dim=1), dims=[3])
                probs = probs / 2
        conf, pred = probs.max(dim=1)
        pred = torch.where(conf < threshold, torch.zeros_like(pred), pred)
        pred_np = pred.cpu().numpy()
        y_np = y.numpy()
        for i in range(pred_np.shape[0]):
            cm += M.confusion_matrix(pred_np[i], y_np[i], C.NUM_CLASSES)
            if len(bf1) < 32:                        # boundary F1 on first 32 imgs (cost)
                bf1.append(M.boundary_f1(pred_np[i], y_np[i], C.NUM_CLASSES))
    out = M.metrics_from_confusion(cm)
    out['boundary_f1'] = float(np.mean(bf1)) if bf1 else float('nan')
    return out


@torch.no_grad()
def threshold_sweep(model, loader, grid=None):
    """One validation pass that accumulates a confusion matrix per candidate
    threshold - the search space the threshold agent optimises over."""
    grid = grid or C.THRESHOLD_GRID
    dev = _device()
    model = model.to(dev).eval()
    cms = {t: np.zeros((C.NUM_CLASSES, C.NUM_CLASSES), dtype=np.int64) for t in grid}
    for x, y in loader:
        x = x.to(dev)
        with torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
            probs = F.softmax(model(x).float(), dim=1)
        conf, pred = probs.max(dim=1)
        for t in grid:
            p = torch.where(conf < t, torch.zeros_like(pred), pred).cpu().numpy()
            for i in range(p.shape[0]):
                cms[t] += M.confusion_matrix(p[i], y.numpy()[i], C.NUM_CLASSES)
    return [{'threshold': t, **M.metrics_from_confusion(cms[t])} for t in grid]


@torch.no_grad()
def save_worst_overlays(model, loader, class_names, out_dir, k=8):
    """Failure-case gallery: the k lowest-IoU images, saved as overlays."""
    from data_viz import overlay
    dev = _device()
    model = model.to(dev).eval()
    scored = []
    for bi, (x, y) in enumerate(loader):
        xd = x.to(dev)
        with torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
            pred = model(xd).float().argmax(dim=1).cpu().numpy()
        y_np = y.numpy()
        for i in range(pred.shape[0]):
            m = M.metrics_from_confusion(M.confusion_matrix(pred[i], y_np[i], C.NUM_CLASSES))
            scored.append((m['miou'], bi, i, x[i].numpy(), y_np[i], pred[i]))
    scored.sort(key=lambda r: r[0])
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for rank, (miou, bi, i, x_np, y_np, p_np) in enumerate(scored[:k]):
        p = os.path.join(out_dir, f'failure_{rank}_miou{miou:.3f}.png')
        overlay(x_np, y_np, p_np, class_names, p)
        paths.append(p)
    return paths

In [ ]:
%%writefile /kaggle/working/code/report_utils.py
"""Reading and writing the results/ folder. Every experiment saves a
small JSON here; the report and slides are generated from these files,
so no number in the report is typed by hand (or invented)."""

import os
import json
import glob
import numpy as np

import config as C


def save_result(name, payload, results_dir=None):
    results_dir = results_dir or C.RESULTS_DIR
    os.makedirs(results_dir, exist_ok=True)
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2, default=_jsonable)
    return path


def load_result(name, results_dir=None):
    results_dir = results_dir or C.RESULTS_DIR
    path = os.path.join(results_dir, f'{name}.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def load_all(results_dir=None):
    results_dir = results_dir or C.RESULTS_DIR
    out = {}
    for p in sorted(glob.glob(os.path.join(results_dir, '*.json'))):
        name = os.path.splitext(os.path.basename(p))[0]
        with open(p) as f:
            out[name] = json.load(f)
    return out


def mean_std_over_seeds(records, key):
    """records: list of per-seed dicts -> 'mean +/- std' (and the raw list)."""
    vals = [r[key] for r in records if r and r.get(key) is not None]
    if not vals:
        return None
    return {'mean': float(np.mean(vals)), 'std': float(np.std(vals)),
            'n_seeds': len(vals), 'values': [float(v) for v in vals]}


def status(results_dir=None):
    """Which result files exist and which are still missing - printed by the
    notebooks and used to stop the report silently using placeholders."""
    results_dir = results_dir or C.RESULTS_DIR
    expected = (['models', 'dataset_stats', 'baseline_slic_rf', 'config']
                + [f'loss_sweep_{m}' for m in C.MODELS]
                + [f'agent_threshold_choice_{m}' for m in C.MODELS]
                + [f'agent_weight_choice_{m}' for m in C.MODELS]
                + [f'ablations_{m}' for m in C.MODELS]
                + [f'test_final_{m}' for m in C.MODELS])
    have = {os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(os.path.join(results_dir, '*.json'))}
    return {name: (name in have) for name in expected}


def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

**What you should see:** the folder cell prints `folder ready: True`, then each of the 7 `%%writefile` cells prints `Writing /kaggle/working/code/<name>.py`. If any prints an error instead, stop and send the error.


### Part 2 file: the agents (new)
The next cell writes `agent_search.py`, your project's agent file adapted for Kaggle (resumable search, the default weights as trial 0). **What you should see:** `Writing /kaggle/working/code/agent_search.py`.

In [ ]:
%%writefile /kaggle/working/code/agent_search.py
"""Agent-based selection of the composite-loss weights and the confidence
threshold (Part C). This is the project's agent_search.py, adapted for Kaggle:

 * the Optuna study is stored in a small database file in results/, so a stopped
   session resumes instead of restarting
 * trial 0 is always the DEFAULT weights (equal weights), trained with exactly
   the same proxy budget as every agent-sampled trial, so "default vs agent
   selected" is measured under identical conditions. It counts toward the
   trial budget (config.AGENT_N_TRIALS trials in total, default included)
 * TPE uses n_startup_trials=4 random trials, then models the results. (Optuna's
   own default is 10 random trials, which would make a 10-trial budget pure
   random search.)
Only train and validation images are used. Test is never touched here.
"""
import os
import csv
import json
import time
import numpy as np

import config as C

N_STARTUP_TRIALS = 4


def threshold_agent(sweep_rows, model_name, results_dir=None):
    """Exhaustive search over config.THRESHOLD_GRID on validation mIoU."""
    results_dir = results_dir or C.RESULTS_DIR
    os.makedirs(results_dir, exist_ok=True)
    log_path = os.path.join(results_dir, f'agent_threshold_trials_{model_name}.csv')
    keys = ['threshold', 'miou', 'mean_dice', 'mean_precision', 'mean_recall', 'pixel_accuracy']
    with open(log_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in sweep_rows:
            w.writerow({k: r.get(k) for k in keys})
    best = max(sweep_rows, key=lambda r: (r['miou'], -abs(r['threshold'])))
    default = next(r for r in sweep_rows if r['threshold'] == 0.0)
    choice = {'model': model_name, 'objective': 'val mIoU',
              'strategy': 'exhaustive grid search (agent)',
              'search_space': [r['threshold'] for r in sweep_rows],
              'n_trials': len(sweep_rows),
              'chosen_threshold': best['threshold'],
              'chosen_val_miou': best['miou'],
              'default_threshold': 0.0,
              'default_val_miou': default['miou'],
              'log_csv': log_path}
    with open(os.path.join(results_dir, f'agent_threshold_choice_{model_name}.json'), 'w') as f:
        json.dump(choice, f, indent=2)
    return choice


def _is_default(trial, names):
    vals = [trial.params.get(f'lambda_{k}') for k in names]
    return all(v is not None and abs(v - vals[0]) < 1e-12 for v in vals)


def composite_weight_agent(model_name, component_names, make_model_and_loaders,
                           results_dir=None, n_trials=None, seed=None, stop_check=None):
    import optuna
    import torch
    import engine
    import losses
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    TS = optuna.trial.TrialState
    results_dir = results_dir or C.RESULTS_DIR
    os.makedirs(results_dir, exist_ok=True)
    n_trials = n_trials or C.AGENT_N_TRIALS
    seed = C.AGENT_SEED if seed is None else seed
    db = os.path.join(results_dir, f'agent_weight_study_{model_name}.db')
    log_path = os.path.join(results_dir, f'agent_weight_trials_{model_name}.csv')
    names = list(component_names)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed, n_startup_trials=N_STARTUP_TRIALS),
        study_name=f'composite_weights_{model_name}',
        storage=f'sqlite:///{db}', load_if_exists=True)
    for t in study.trials:                      # a trial cut off by a stopped session
        if t.state == TS.RUNNING:
            try:
                study.tell(t.number, state=TS.FAIL)
            except Exception as e:              # noqa: BLE001
                print('could not close stale trial', t.number, e)
    for leftover in os.listdir(C.CKPT_DIR):     # free disk from cut-off trials
        if f'{model_name}_agent_' in leftover and leftover.endswith('_ckpt.pt'):
            os.remove(os.path.join(C.CKPT_DIR, leftover))

    def run_trial(lambdas, tag, trial_seed):
        model, tl, vl = make_model_and_loaders()
        loss_fn = losses.CompositeLoss(
            {k: losses.build_loss(k, C.NUM_CLASSES) for k in names}, lambdas)
        t0 = time.time()
        history = engine.train(model, tag, tl, vl, loss_fn, epochs=C.EPOCHS_TRIAL,
                               seed=trial_seed, verbose=False)
        secs = time.time() - t0
        p = engine.ckpt_path(tag)
        if os.path.exists(p):
            os.remove(p)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return history[-1]['val_miou'], secs

    def n_done():
        return len([t for t in study.trials if t.state == TS.COMPLETE])

    # trial 0 = the default (equal) weights, same proxy budget as all other trials
    if stop_check and stop_check():
        print('[stop] not enough session time left to start the agent. Run again in a new session.')
        return None
    if not any(_is_default(t, names) for t in study.trials if t.state == TS.COMPLETE):
        lam = {k: 1.0 / len(names) for k in names}
        val, secs = run_trial(lam, f'{model_name}_agent_default', seed)
        dist = {f'lambda_{k}': optuna.distributions.FloatDistribution(0.0, 1.0) for k in names}
        study.add_trial(optuna.trial.create_trial(
            params={f'lambda_{k}': 1.0 for k in names}, distributions=dist, value=val,
            user_attrs={'seconds': secs}))
        print(f'{model_name} agent trial (default equal weights): val mIoU {val:.4f} ({secs:.0f}s)')

    def objective(trial):
        raw = {k: trial.suggest_float(f'lambda_{k}', 0.0, 1.0) for k in names}
        total = sum(raw.values()) or 1.0
        lam = {k: v / total for k, v in raw.items()}
        val, secs = run_trial(lam, f'{model_name}_agent_trial{trial.number}', seed + trial.number)
        trial.set_user_attr('seconds', secs)
        print(f'{model_name} agent trial {trial.number}: weights '
              f'{ {k: round(v, 3) for k, v in lam.items()} } val mIoU {val:.4f} ({secs:.0f}s)')
        return val

    while n_done() < n_trials:
        if stop_check and stop_check():
            print(f'[stop] not enough session time left; {n_done()}/{n_trials} trials done. Run again to continue.')
            break
        study.optimize(objective, n_trials=1)

    rows = []
    for t in study.trials:
        if t.state != TS.COMPLETE:
            continue
        raw = {k: t.params[f'lambda_{k}'] for k in names}
        total = sum(raw.values()) or 1.0
        rows.append({'trial': t.number, 'is_default': _is_default(t, names), 'val_miou': t.value,
                     **{f'weight_{k}': raw[k] / total for k in names},
                     'seconds': t.user_attrs.get('seconds')})
    if rows:
        with open(log_path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
    if n_done() < n_trials:
        return None
    best = study.best_trial
    raw = {k: best.params[f'lambda_{k}'] for k in names}
    total = sum(raw.values()) or 1.0
    default_row = next(r for r in rows if r['is_default'])
    choice = {'model': model_name, 'objective': 'val mIoU (proxy: subset training, subset validation)',
              'strategy': f'Bayesian (Optuna TPE, seed {seed}, {N_STARTUP_TRIALS} random start-up trials)',
              'components': names,
              'n_trials': n_trials,
              'default_trial_included_in_budget': True,
              'compute_budget_per_trial': f'{C.AGENT_TRAIN_SUBSET} train images, {C.EPOCHS_TRIAL} epochs, '
                                          f'300 validation images',
              'stopping_rule': 'fixed trial budget',
              'chosen_trial': best.number,
              'chosen_weights': {k: v / total for k, v in raw.items()},
              'chosen_val_miou': best.value,
              'default_weights': {k: 1.0 / len(names) for k in names},
              'default_val_miou': default_row['val_miou'],
              'chosen_is_default': best.number == default_row['trial'],
              'log_csv': log_path}
    with open(os.path.join(results_dir, f'agent_weight_choice_{model_name}.json'), 'w') as f:
        json.dump(choice, f, indent=2)
    return choice


In [ ]:
# CELL 2: check that all 8 code files were written before continuing
import os
need = ['config.py', 'metrics.py', 'losses.py', 'models.py', 'data.py', 'engine.py', 'report_utils.py', 'agent_search.py']
missing = [f for f in need if not os.path.exists('/kaggle/working/code/' + f)]
if missing:
    raise FileNotFoundError('missing code files: ' + ', '.join(missing) +
        '. Fix: run the cells in order starting from "# CELL 1b: create the folder the code files are written into" down to the agent_search.py cell, then run this cell again.')
print('code files ready: 8/8')


In [ ]:
# CELL 3: load the project code and print the settings (defaults are safe, nothing to change)
import sys, os, time, json
sys.path.insert(0, '/kaggle/working/code')
import config as C
import data, engine, losses, models, report_utils
C.ensure_dirs()
print('settings ok | models:', C.MODELS, '| main epochs:', C.EPOCHS_MAIN, '| sweep epochs:', C.EPOCHS_SWEEP)


**expected output:** `settings ok | models: ['deeplabv3plus', 'segformer_b3'] | main epochs: 12 | sweep epochs: 6`.

In [ ]:
# CELL 4: find dataset, copy images to fast local disk, restore checkpoints and results from the inputs
import glob, shutil, torch
inputs = '/kaggle/input'
roots = sorted({os.path.dirname(p) for p in glob.glob(inputs + '/**/masks_index.csv', recursive=True)})
assert roots, 'Dataset not found.'
SRC = roots[0]
print('dataset folder:', SRC)

# 1) copy images + masks to local disk (training from the mounted input folder is slower)
LOCAL = C.LOCAL_DS
if not os.path.exists(os.path.join(LOCAL, 'masks_index.csv')):
    t0 = time.time()
    for sub in ['images', 'masks']:
        shutil.copytree(os.path.join(SRC, sub), os.path.join(LOCAL, sub), dirs_exist_ok=True)
    import pandas as pd
    idx = pd.read_csv(os.path.join(SRC, 'masks_index.csv'))
    idx['image_path'] = LOCAL + '/' + idx['image_path']
    idx['mask_path'] = LOCAL + '/' + idx['mask_path']
    idx.to_csv(C.MASKS_INDEX, index=False)
    print(f'copied to {LOCAL} in {time.time()-t0:.0f}s')
else:
    print('local copy already there')

# 2) restore checkpoints/results from ALL attached inputs; if a file exists in several,
#    keep the most advanced one (highest epoch for checkpoints, biggest file for json/weights)
def walk_files(root):
    for dp, dn, fn in os.walk(root):
        dn[:] = [d for d in dn if d not in ('images', 'masks')]
        for f in fn:
            yield os.path.join(dp, f)
best = {}
for p in walk_files(inputs):
    f = os.path.basename(p)
    if f.endswith('_ckpt.pt'):
        score = torch.load(p, map_location='cpu', weights_only=False)['epoch']
        dest = os.path.join(C.CKPT_DIR, f)
    elif f.endswith('_final.pt'):
        score, dest = os.path.getsize(p), os.path.join(C.CKPT_DIR, f)
    elif f.endswith('.json') and os.sep + 'results' + os.sep in p and os.sep + 'figures' + os.sep not in p:
        score, dest = os.path.getsize(p), os.path.join(C.RESULTS_DIR, f)
    else:
        continue
    if dest not in best or score > best[dest][0]:
        best[dest] = (score, p)
for dest, (score, p) in best.items():
    if not os.path.exists(dest):
        shutil.copyfile(p, dest)
        print('restored', os.path.basename(dest), '<-', p.split('/')[3] if p.count('/') > 3 else p)
frames = data.load_frames()
print('frames:', {s: len(f) for s, f in frames.items()})
for f in sorted(glob.glob(C.CKPT_DIR + '/*_ckpt.pt')):
    print('checkpoint', os.path.basename(f), 'is at epoch', torch.load(f, map_location='cpu', weights_only=False)['epoch'] + 1)
CW = report_utils.load_result('class_weights')['median_frequency_weights']
print('class weights loaded:', len(CW), 'values')

**expected output** `dataset folder: /kaggle/input/...`, `copied to /tmp/ds in ...s`, several `restored ...` lines, `frames: {'train': 8000, 'val': 1000, 'test': 1000}` and `checkpoint segformer_b3_main_ce_ckpt.pt is at epoch 5`

In [ ]:
# PART 2 - CELL P2: settings and required-input check
# v7 REPLACEMENT: same settings/provenance/restore as before, plus a STRONGER preflight:
# the finished DeepLabV3+ seed runs and the SegFormer-B3 seed-42 weights from the earlier session
# are now REQUIRED inputs, so a missing file stops the notebook here instead of retraining anything.
# settings, extra install, provenance log, restore extra files, preflight check (stops with a clear error if something required is missing)
import subprocess, sys, os, glob, json, time, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'], check=True)
import numpy as np, torch, optuna, transformers, segmentation_models_pytorch as smp
import torch.nn.functional as F
sys.path.insert(0, '/kaggle/working/code')
import importlib, agent_search
importlib.reload(agent_search)

# settings (the defaults keep every comparison like-for-like)
PART2_MODELS = ['deeplabv3plus', 'segformer_b3']     # remove one to run a model in a separate session
COMPONENTS = ['ce', 'dice', 'lovasz']                 # composite = weighted sum of these three losses
EPOCHS_FINAL = C.EPOCHS_SWEEP                         # 6: same budget as the loss-comparison runs 
MAX_HOURS_2 = 8.5                                     # stop cleanly before the shortest per-session limit Kaggle documents (9 hours)
T0_2 = time.time()
assert EPOCHS_FINAL == C.EPOCHS_SWEEP, 'Part 2 is built around the 6-epoch budget (matches existing CE, aug-off and class-weight runs)'

# provenance: what this session really ran on 
def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:                            # noqa: BLE001
        return f'unavailable: {e}'
prov = {'session_started': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'nvidia_smi': sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'),
        'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': transformers.__version__,
        'smp': smp.__version__, 'optuna': optuna.__version__,
        'part2_settings': {'models': PART2_MODELS, 'components': COMPONENTS, 'epochs_final': EPOCHS_FINAL,
                           'seeds': C.SEEDS, 'agent_trials': C.AGENT_N_TRIALS,
                           'agent_train_subset': C.AGENT_TRAIN_SUBSET, 'epochs_trial': C.EPOCHS_TRIAL,
                           'threshold_grid': C.THRESHOLD_GRID}}
log = report_utils.load_result('provenance_part2') or {'sessions': []}
log['sessions'].append(prov)
report_utils.save_result('provenance_part2', log)
print('GPU (from nvidia-smi):', prov['nvidia_smi'])

# ---------------- restore extra result files (CELL 4 only restores .json and checkpoints) ----------------
for pat in ['agent_weight_study_*.db', 'agent_weight_trials_*.csv', 'agent_threshold_trials_*.csv']:
    for p in glob.glob('/kaggle/input/**/results/' + pat, recursive=True):
        dest = os.path.join(C.RESULTS_DIR, os.path.basename(p))
        if not os.path.exists(dest) or os.path.getsize(p) > os.path.getsize(dest):
            shutil.copyfile(p, dest); print('restored', os.path.basename(p))

# ---------------- preflight: REQUIRED inputs. Anything missing stops the notebook here with what to fix ----------------
errors = []
if not torch.cuda.is_available():
    errors.append('No GPU. Session options > Accelerator > choose a GPU, then restart the session and run again.')
print('\npreflight (True = found)')
for name in PART2_MODELS:
    sweep = report_utils.load_result(f'loss_sweep_{name}') or {}
    need = {
        f'loss_sweep_{name}.json with the "ce" and "weighted_ce" runs': ('ce' in sweep and 'weighted_ce' in sweep),
        f'{name}_sweep_ce_final.pt (the weights of our 6-epoch CE run)': os.path.exists(os.path.join(C.CKPT_DIR, f'{name}_sweep_ce_final.pt')),
        f'{name}_sweep_weighted_ce_final.pt (the weights of our 6-epoch class-weighted CE run)': os.path.exists(os.path.join(C.CKPT_DIR, f'{name}_sweep_weighted_ce_final.pt')),
    }
    if name == 'segformer_b3':
        need['ablations_segformer_b3.json with "augmentation_off"'] = 'augmentation_off' in (report_utils.load_result('ablations_segformer_b3') or {})
        need['segformer_b3_final_seed42_final.pt (the composite seed-42 weights from the session that asked for the review; kept as evidence)'] = \
            os.path.exists(os.path.join(C.CKPT_DIR, 'segformer_b3_final_seed42_final.pt'))
    if name == 'deeplabv3plus':
        for s in C.SEEDS:
            need[f'deeplabv3plus_final_seed{s}_final.pt (finished DeepLabV3+ run from the earlier session - must NOT be retrained)'] = \
                os.path.exists(os.path.join(C.CKPT_DIR, f'deeplabv3plus_final_seed{s}_final.pt'))
        need['test_final_deeplabv3plus.json complete (the finished DeepLabV3+ test result)'] = \
            bool((report_utils.load_result('test_final_deeplabv3plus') or {}).get('complete'))
    for k, ok in need.items():
        print(f'  {name:14s} {k}: {ok}')
        if not ok:
            errors.append(f'Missing for {name}: {k}.')
    print(f'  {name:14s} progress already saved: agent_weights={bool(report_utils.load_result("agent_weight_choice_" + name))} '
          f'seed_runs_done={[os.path.exists(os.path.join(C.CKPT_DIR, f"{name}_final_seed{s}_final.pt")) for s in C.SEEDS]}')
if errors:
    raise RuntimeError('STOP - required input missing:\n - ' + '\n - '.join(errors))
print('\nPREFLIGHT PASSED: every required earlier result is present.')

**P2 - settings and required-input check**

Installs `optuna`, prints `GPU (from nvidia-smi): ...` (the recorded GPU name for the report), then one line per required earlier result. Ends with `PREFLIGHT PASSED`. If something required is missing it stops with a `RuntimeError` that says what to attach.

In [ ]:
# PART 2 - CELL P3: helper functions
# helper functions (defines functions only, trains nothing)
def hours_left(): return MAX_HOURS_2 - (time.time() - T0_2) / 3600

def ckpt_final(tag): return os.path.join(C.CKPT_DIR, f'{tag}_final.pt')

def est_epoch_seconds(name):
    """Seconds per epoch measured in OUR 6-epoch CE run of this model (used only to decide if time is left)."""
    h = ((report_utils.load_result(f'loss_sweep_{name}') or {}).get('ce') or {}).get('history') or []
    secs = [e['epoch_seconds'] for e in h]
    return float(np.mean(secs)) if secs else None

def val_loader_for(name, subset=None):
    return data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name], subset=subset)

def load_model(name, path):
    m = models.build_model(name)
    m.load_state_dict(torch.load(path, map_location='cpu'))
    return m

def composite_loss(weights, rebalanced=False):
    """rebalanced=True swaps the plain CE term for class-weighted CE (median-frequency weights from class_weights.json)."""
    comps = {k: losses.build_loss('weighted_ce' if (rebalanced and k == 'ce') else k, C.NUM_CLASSES, CW) for k in weights}
    return losses.CompositeLoss(comps, weights)

def train_run(name, tag, loss_fn, seed, epochs=None, aug=True):
    """Train one run (resumes after a stop), save weights + a result file, delete the big resume file."""
    epochs = epochs or EPOCHS_FINAL
    final = ckpt_final(tag)
    res = report_utils.load_result(f'run_{tag}')
    vl = val_loader_for(name)
    if os.path.exists(final) and res:
        print(f'[skip] {tag} already trained (val mIoU {res["val"]["miou"]:.4f})'); return res
    if os.path.exists(final):                         # weights saved but result file missing: just re-evaluate
        model = load_model(name, final)
        res = {'tag': tag, 'seed': seed, 'epochs': epochs, 'augment': aug, 'history': None,
               'val': engine.evaluate(model, vl), 'note': 're-evaluated from saved weights'}
        report_utils.save_result(f'run_{tag}', res); del model; return res
    est = est_epoch_seconds(name)
    need_h = (est or 0) * epochs / 3600
    if not os.path.exists(engine.ckpt_path(tag)) and hours_left() < need_h * 1.15:
        print(f'[stop] {hours_left():.1f} h left, {tag} needs about {need_h:.1f} h (estimate). Save the version and run again.')
        return None
    tl = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=aug)
    if hasattr(loss_fn, 'to'): loss_fn = loss_fn.to('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'=== {tag}: {epochs} epochs, seed {seed}, augmentation {"on" if aug else "off"} ===')
    model = models.build_model(name)
    history = engine.train(model, tag, tl, vl, loss_fn, epochs=epochs, seed=seed)
    val = engine.evaluate(model, vl)
    torch.save(model.state_dict(), final)
    res = {'tag': tag, 'model': name, 'seed': seed, 'epochs': epochs, 'augment': aug, 'history': history, 'val': val,
           'seconds_per_epoch': float(np.mean([h['epoch_seconds'] for h in history])),
           'loss_weights': getattr(loss_fn, 'weights', None)}
    report_utils.save_result(f'run_{tag}', res)
    p = engine.ckpt_path(tag)
    if os.path.exists(p): os.remove(p)
    del model; torch.cuda.empty_cache()
    print(f'{tag} val mIoU: {val["miou"]:.4f}')
    return res
print('helpers ready | hours left:', round(hours_left(), 1))


**P3 - helper functions**

Prints `helpers ready | hours left: ...`. Nothing is trained.

In [ ]:
# PART 2 - CELL P4: smoke test
# SMOKE TEST. A few minutes, trains nothing, writes nothing into results/. Stops the notebook if anything is broken.
import tempfile, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import metrics as M
problems = []
def check(label, fn):
    try:
        out = fn(); print('OK  ', label, out if out else '')
    except Exception as e:                                           # noqa: BLE001
        problems.append(label); print('FAIL', label, '|', type(e).__name__, str(e)[:300])
tmp = tempfile.mkdtemp()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def disk():
    free = shutil.disk_usage('/kaggle/working').free / 1e9
    assert free >= 12, f'only {free:.1f} GB free in /kaggle/working, about 12 GB are needed for checkpoints'
    return f'{free:.0f} GB free'
check('disk space', disk)

def results_io():
    report_utils.save_result('smoke', {'a': 1}, results_dir=tmp)
    assert report_utils.load_result('smoke', results_dir=tmp) == {'a': 1}
check('results read/write', results_io)

def optuna_db():
    st = optuna.create_study(direction='maximize', storage=f'sqlite:///{tmp}/s.db', study_name='s', load_if_exists=True,
                             sampler=optuna.samplers.TPESampler(seed=1, n_startup_trials=agent_search.N_STARTUP_TRIALS))
    d = {'lambda_ce': optuna.distributions.FloatDistribution(0, 1)}
    st.add_trial(optuna.trial.create_trial(params={'lambda_ce': 1.0}, distributions=d, value=0.5))
    tr = st.ask(); tr.suggest_float('lambda_ce', 0, 1)
    st.tell(tr.number, state=optuna.trial.TrialState.FAIL)
    st2 = optuna.load_study(study_name='s', storage=f'sqlite:///{tmp}/s.db')
    assert len(st2.trials) == 2
check('optuna sqlite study (create, add default trial, close stale trial, reload)', optuna_db)

def figs():
    fig, ax = plt.subplots(); ax.imshow(np.zeros((3, 3))); fig.savefig(f'{tmp}/f.png'); plt.close(fig)
check('matplotlib figure saving', figs)

for name in PART2_MODELS:
    def one_model(name=name):
        wc = {'ce': 1/3, 'dice': 1/3, 'lovasz': 1/3}
        tl = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True, subset=2 * C.BATCH_SIZE[name])
        vl = data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name], subset=16)
        model = models.build_model(name).to(dev).train()
        x, y = next(iter(tl)); x, y = x.to(dev), y.to(dev)
        for rebalanced in (False,):
            lf = composite_loss(wc, rebalanced).to(dev)
            with torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
                loss = lf(model(x), y)
            loss.backward(); model.zero_grad(set_to_none=True)
            assert torch.isfinite(loss).item(), 'loss is not finite'
        m = engine.evaluate(model, vl, threshold=0.3, tta=True, max_batches=2)
        rows = engine.threshold_sweep(model, data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name], subset=8))
        assert len(rows) == len(C.THRESHOLD_GRID)
        agent_search.threshold_agent(rows, f'smoke_{name}', results_dir=tmp)
        mem = torch.cuda.max_memory_allocated() / 1e9 if dev == 'cuda' else 0
        del model; torch.cuda.empty_cache()
        return f'forward/backward, composite loss, evaluate(threshold, TTA), threshold sweep + agent all ran (peak GPU memory {mem:.1f} GB)'
    check(f'{name}: one real batch through the whole pipeline', one_model)

shutil.rmtree(tmp, ignore_errors=True)
if problems:
    raise RuntimeError('SMOKE TEST FAILED: ' + '; '.join(problems) + '. Nothing was trained.')
print('\nSMOKE TEST PASSED. Training can start.')


**P4 - smoke test**

Prints `OK` lines (disk, results files, Optuna database, figures, then one real batch per model through the composite loss, evaluation with threshold and TTA, and the threshold agent). Ends with `SMOKE TEST PASSED`. Any `FAIL` stops the notebook with nothing trained.

In [ ]:
# PART 2 - CELL P5: weight agent
# STEP 1 - the composite-weight agent (Optuna), once per model. Validation data only.
for name in PART2_MODELS:
    if report_utils.load_result(f'agent_weight_choice_{name}'):
        print(f'[skip] weight agent for {name} already done'); continue
    def make_model_and_loaders(name=name):
        m = models.build_model(name)
        tl = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True,
                             subset=C.AGENT_TRAIN_SUBSET, seed=C.AGENT_SEED)
        vl = data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name], subset=300, seed=C.AGENT_SEED)
        return m, tl, vl
    print(f'=== weight agent for {name}: {C.AGENT_N_TRIALS} trials (trial 0 = default equal weights) ===')
    choice = agent_search.composite_weight_agent(name, COMPONENTS, make_model_and_loaders,
                                                 stop_check=lambda: hours_left() < 0.5)
    if choice:
        print(f'{name} chosen weights: { {k: round(v, 3) for k, v in choice["chosen_weights"].items()} } | '
              f'proxy val mIoU {choice["chosen_val_miou"]:.4f} vs default {choice["default_val_miou"]:.4f}')
print('hours left:', round(hours_left(), 1))


**P5 - weight agent**

Per model, one line per trial with the three weights and validation mIoU (trial 0 is the default equal weights), then `chosen weights ... vs default ...`. Resumes from saved trials after a stop.

In [ ]:
# PART 2 - CELL P6: final configuration, seed 42
# STEP 2 - final configuration, seed 42 (composite loss with the agent weights), one run per model
for name in PART2_MODELS:
    wc = report_utils.load_result(f'agent_weight_choice_{name}')
    if not wc:
        print(f'[wait] {name}: the weight agent has not finished, run cell P5 first'); continue
    train_run(name, f'{name}_final_seed{C.SEEDS[0]}', composite_loss(wc['chosen_weights']), seed=C.SEEDS[0])
print('hours left:', round(hours_left(), 1))


**P6 - final configuration, seed 42**

Six epoch lines per model, then the validation mIoU. This is seed 1 of 3.

In [ ]:
# PART 2 - CELL P7: threshold agent
# STEP 3 - the threshold agent on the seed-42 composite model (validation only)
for name in PART2_MODELS:
    if report_utils.load_result(f'agent_threshold_choice_{name}'):
        print(f'[skip] threshold agent for {name} already done'); continue
    path = ckpt_final(f'{name}_final_seed{C.SEEDS[0]}')
    if not os.path.exists(path):
        print(f'[wait] {name}: seed-42 final model not trained yet (cell P6)'); continue
    model = load_model(name, path)
    rows = engine.threshold_sweep(model, val_loader_for(name))
    ch = agent_search.threshold_agent(rows, name)
    print(f'{name}: default threshold 0.0 -> val mIoU {ch["default_val_miou"]:.4f} | agent chose {ch["chosen_threshold"]} -> {ch["chosen_val_miou"]:.4f}')
    del model; torch.cuda.empty_cache()


**P7 - threshold agent**

`default threshold 0.0 -> val mIoU ... | agent chose ... -> ...` per model.

In [ ]:
# PART 2 - CELL P8: DeepLabV3+ without augmentation
# The one remaining training run for the ablation tables, done before the freeze gate: DeepLabV3+ with augmentation OFF.
# Same loss (CE), 6-epoch budget and seed 42 as our existing 'ce' run, so the only difference is augmentation.
# SegFormer-B3 already has this ablation in ablations_segformer_b3.json.
if 'deeplabv3plus' in PART2_MODELS:
    train_run('deeplabv3plus', 'deeplabv3plus_ablation_noaug', losses.build_loss('ce', C.NUM_CLASSES),
              seed=C.SEEDS[0], epochs=C.EPOCHS_SWEEP, aug=False)
print('hours left:', round(hours_left(), 1))


**P8 - remaining ablation runs**

Epoch lines for one 6-epoch run (DeepLabV3+ without augmentation). Resume-safe.

In [ ]:
# PART 2 - CELL P9: ablation tables
# STEP 5 - ablation tables on VALIDATION (seed 42, 6 epochs): improvements one at a time, and added cumulatively. Evaluation only, no training.
def aug_off_val(name):
    if name == 'segformer_b3':
        r = (report_utils.load_result('ablations_segformer_b3') or {}).get('augmentation_off')
        return r['val'] if r else None
    r = report_utils.load_result('run_deeplabv3plus_ablation_noaug')
    return r['val'] if r else None

for name in PART2_MODELS:
    out_name = f'ablation_ladder_{name}'
    if report_utils.load_result(out_name):
        print(f'[skip] {out_name} already written'); continue
    tc = report_utils.load_result(f'agent_threshold_choice_{name}')
    wc = report_utils.load_result(f'agent_weight_choice_{name}')
    sweep = report_utils.load_result(f'loss_sweep_{name}')
    comp_path = ckpt_final(f'{name}_final_seed{C.SEEDS[0]}')
    base_path = os.path.join(C.CKPT_DIR, f'{name}_sweep_ce_final.pt')
    wce_path = os.path.join(C.CKPT_DIR, f'{name}_sweep_weighted_ce_final.pt')
    off = aug_off_val(name)
    miss = [n for n, ok in [('agent threshold', tc), ('agent weights', wc), ('seed-42 composite weights', os.path.exists(comp_path)),
                            ('augmentation-off result', off)] if not ok]
    if miss:
        print(f'[wait] {name}: ablation table needs {miss}. Finish the earlier cells (P5-P8).'); continue
    vl = val_loader_for(name)
    rows = {}
    # --- chain A: the CE family, improvements added cumulatively (all evaluation of existing weights) ---
    rows['A0 no augmentation (CE)'] = off
    rows['A1 + augmentation = baseline CE (plain argmax, no TTA)'] = sweep['ce']['val']
    rows['A2 + class rebalancing (weighted CE, plain argmax)'] = sweep['weighted_ce']['val']
    model = load_model(name, wce_path)
    w_choice = agent_search.threshold_agent(engine.threshold_sweep(model, vl), f'{name}_rebalanced')
    rows['A3 + agent threshold (chosen for the rebalanced model)'] = engine.evaluate(model, vl, threshold=w_choice['chosen_threshold'])
    rows['A4 + hflip TTA'] = engine.evaluate(model, vl, threshold=w_choice['chosen_threshold'], tta=True)
    del model; torch.cuda.empty_cache()
    # --- one at a time on the CE baseline ---
    model = load_model(name, base_path)
    b_choice = agent_search.threshold_agent(engine.threshold_sweep(model, vl), f'{name}_ce_baseline')
    rows['S1 baseline CE + agent threshold only'] = engine.evaluate(model, vl, threshold=b_choice['chosen_threshold'])
    rows['S2 baseline CE + hflip TTA only'] = engine.evaluate(model, vl, tta=True)
    recheck = engine.evaluate(model, vl)
    del model; torch.cuda.empty_cache()
    # --- chain B: composite loss with agent weights (seed 42), then agent threshold, then TTA = final configuration ---
    model = load_model(name, comp_path)
    thr = tc['chosen_threshold']
    rows['B1 baseline CE + agent composite weights'] = engine.evaluate(model, vl)
    rows['B2 + agent threshold'] = engine.evaluate(model, vl, threshold=thr)
    rows['B3 + hflip TTA  (= FINAL CONFIGURATION)'] = engine.evaluate(model, vl, threshold=thr, tta=True)
    del model; torch.cuda.empty_cache()
    b = sweep['ce']['val']['miou']
    report_utils.save_result(out_name, {
        'model': name, 'split': 'validation', 'seed': C.SEEDS[0], 'epochs': EPOCHS_FINAL,
        'agent_weights': wc['chosen_weights'], 'agent_threshold_composite': thr,
        'agent_threshold_rebalanced': w_choice['chosen_threshold'], 'agent_threshold_ce_baseline': b_choice['chosen_threshold'],
        'weights_default_vs_agent_at_PROXY_budget': {'default_equal_weights_val_miou': wc['default_val_miou'],
            'agent_chosen_val_miou': wc['chosen_val_miou'], 'chosen_is_default': wc['chosen_is_default'],
            'budget': wc['compute_budget_per_trial'], 'note': 'proxy budget only (not a full-budget comparison)'},
        'threshold_default_vs_agent_full_model': {'default_0.0': tc['default_val_miou'], 'agent_chosen': tc['chosen_val_miou']},
        'baseline_recheck_miou': recheck['miou'], 'baseline_stored_miou': b,
        'rows': {k: {'miou': r['miou'], 'metrics': r} for k, r in rows.items()}})
    print(f'\n--- {name}: validation mIoU, seed 42, {EPOCHS_FINAL} epochs ---')
    for k, r in rows.items():
        print(f'  {k:62s} {r["miou"]:.4f}  ({r["miou"] - b:+.4f} vs baseline CE)')
    print(f'  sanity: re-evaluated baseline {recheck["miou"]:.4f} vs stored {b:.4f} (should agree to about 3 decimals)')
    print(f'  agent weights vs default equal weights, PROXY budget only: {wc["chosen_val_miou"]:.4f} vs {wc["default_val_miou"]:.4f}')
    print('  Rows lower than the baseline are real results and stay in the report.')


**P9 - ablation tables**

A table per model: A rows are the cumulative chain on the CE family, S rows are each improvement alone on the CE baseline, B rows are the composite chain ending in the final configuration. Evaluation only. Lower-than-baseline rows are real results. The sanity line should agree to about 3 decimals. Validation only.

In [ ]:
# PART 2 - CELL P10: freeze gate
# v7 REPLACEMENT (2026-11-14, after the freeze-gate REVIEW REQUIRED for SegFormer-B3).
# Gate outcome: composite 0.6740 < plain CE 0.6898, TTA 0.6747 < 0.6760 on validation.
# This version implements a PROPOSED selection rule for SegFormer-B3: freeze the best MEASURED
# validation configuration on the CE weights.
# if the composite configuration should stay instead, do NOT run this version.
# This cell measures the one CE combination never evaluated before (CE + agent threshold + TTA)
# on the VALIDATION set, then freezes the argmax over measured candidates only. No test data is touched.
# DeepLabV3+ is untouched: it froze cleanly in the original run of this cell (skip branch below).
import shutil
REVIEW_DECISION = 'USE_BEST_MEASURED_VALIDATION'      # the proposed rule this version applies; 
fz = report_utils.load_result('final_config_frozen') or {'models': {}}
def row(rows, prefix):
    return next(v['miou'] for k, v in rows.items() if k.startswith(prefix))
for name in PART2_MODELS:
    if fz['models'].get(name, {}).get('status') == 'FROZEN':
        print(f'[skip] {name} final configuration already frozen'); continue
    if name == 'segformer_b3' and REVIEW_DECISION == 'USE_BEST_MEASURED_VALIDATION':
        lad = report_utils.load_result(f'ablation_ladder_{name}')
        if not lad:
            print(f'[wait] {name}: the validation comparisons (cell P9) are not finished'); continue
        R = lad['rows']
        ce_plain = row(R, 'A1')                        # CE, plain argmax          (already measured)
        ce_thr   = row(R, 'S1')                        # CE + agent threshold      (already measured)
        ce_tta   = row(R, 'S2')                        # CE + hflip TTA            (already measured)
        thr_ce   = lad['agent_threshold_ce_baseline']  # the threshold agent's choice for the CE model
        # the only combination not measured anywhere yet: CE + agent threshold + TTA (validation only)
        model = load_model(name, os.path.join(C.CKPT_DIR, f'{name}_sweep_ce_final.pt'))
        ce_both = engine.evaluate(model, val_loader_for(name), threshold=thr_ce, tta=True)['miou']
        del model; torch.cuda.empty_cache()
        cands = {'plain argmax': (ce_plain, 0.0, False), '+ agent threshold': (ce_thr, thr_ce, False),
                 '+ hflip TTA': (ce_tta, 0.0, True), '+ agent threshold + TTA': (ce_both, thr_ce, True)}
        best_name = max(cands, key=lambda k: cands[k][0])
        best_miou, best_thr, best_tta = cands[best_name]
        print(f'--- {name} final-configuration candidates on the CE weights (validation, seed 42, all measured):')
        for k, (m, _, _) in cands.items(): print(f'    CE {k:26s} val mIoU {m:.4f}')
        print(f'  review decision: freeze CE {best_name} (val mIoU {best_miou:.4f}).')
        # preserve the composite seed-42 weights and run record (the report's loss-study evidence)
        comp_ckpt = ckpt_final(f'{name}_final_seed{C.SEEDS[0]}')
        comp_run = report_utils.load_result(f'run_{name}_final_seed{C.SEEDS[0]}')
        if os.path.exists(comp_ckpt) and not os.path.exists(ckpt_final(f'{name}_composite_seed{C.SEEDS[0]}')):
            shutil.copy2(comp_ckpt, ckpt_final(f'{name}_composite_seed{C.SEEDS[0]}'))
        if comp_run and not report_utils.load_result(f'run_{name}_composite_seed{C.SEEDS[0]}'):
            report_utils.save_result(f'run_{name}_composite_seed{C.SEEDS[0]}', comp_run)
        # the CE seed-42 final weights ARE our saved sweep_ce run: identical training configuration
        # (6 epochs, seed 42, augmentation on, plain CE) to the frozen final configuration of this model
        shutil.copy2(os.path.join(C.CKPT_DIR, f'{name}_sweep_ce_final.pt'), comp_ckpt)
        sweep_ce = report_utils.load_result(f'loss_sweep_{name}')['ce']
        report_utils.save_result(f'run_{name}_final_seed{C.SEEDS[0]}', {
            'tag': f'{name}_final_seed{C.SEEDS[0]}', 'model': name, 'seed': C.SEEDS[0], 'epochs': EPOCHS_FINAL,
            'augment': True, 'history': sweep_ce.get('history'), 'val': sweep_ce['val'],
            'seconds_per_epoch': float(np.mean([h['epoch_seconds'] for h in sweep_ce.get('history', [])])) if sweep_ce.get('history') else None,
            'loss_weights': None,
            'note': 'weights are the saved sweep_ce seed-42 run, selected as the final configuration after the freeze-gate review; the composite seed-42 run is preserved as run_segformer_b3_composite_seed42'})
        # the earlier session's error analysis for this model used the composite weights: remove it so P13 recomputes
        ea_path = os.path.join(C.RESULTS_DIR, f'error_analysis_{name}.json')
        if os.path.exists(ea_path): os.remove(ea_path)
        fz['models'][name] = {'status': 'FROZEN', 'weights': None, 'threshold': best_thr, 'tta': best_tta,
            'rebalancing': False, 'augmentation': True, 'loss': 'ce',
            'validation_rows_seed42': {'baseline_CE': ce_plain, 'B1_composite': row(R, 'B1'), 'B2_threshold': row(R, 'B2'), 'B3_tta': row(R, 'B3')},
            'ce_candidates_seed42': {k: v[0] for k, v in cands.items()}, 'selected': f'ce {best_name}',
            'agent_composite_weights_studied': (report_utils.load_result(f'agent_weight_choice_{name}') or {}).get('chosen_weights'),
            'triggers': fz['models'].get(name, {}).get('triggers', []),
            'review_decision': 'USE_BEST_MEASURED_VALIDATION',
            'frozen_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'), 'uses_test': False}
        report_utils.save_result('final_config_frozen', fz)
        continue
    # ---- original gate logic, unchanged, for any model still unfrozen (DeepLabV3+ already froze above) ----
    lad = report_utils.load_result(f'ablation_ladder_{name}')
    tc = report_utils.load_result(f'agent_threshold_choice_{name}')
    wc = report_utils.load_result(f'agent_weight_choice_{name}')
    if not (lad and tc and wc):
        print(f'[HOLD] {name}: the validation comparisons (cell P9) are not finished, so the configuration cannot be frozen and the extra seeds and the test step will not run.'); continue
    R = lad['rows']
    a1, b1, b2, b3 = (row(R, p) for p in ('A1', 'B1', 'B2', 'B3'))
    triggers = []
    if b1 < a1: triggers.append(f'composite loss ({b1:.4f}) is below plain CE ({a1:.4f}) on validation')
    if b2 < b1: triggers.append(f'agent threshold ({b2:.4f}) is below plain argmax ({b1:.4f})')
    if b3 < b2: triggers.append(f'TTA ({b3:.4f}) is below no TTA ({b2:.4f})')
    print(f'--- {name} validation, seed 42: baseline CE {a1:.4f} | + composite {b1:.4f} | + threshold {b2:.4f} | + TTA {b3:.4f}')
    status = 'FROZEN'
    if triggers and REVIEW_DECISION != 'KEEP_FINAL_CONFIGURATION':
        status = 'REVIEW_REQUIRED'
        print('  REVIEW REQUIRED (strict comparison, no tolerance). Validation outcomes that contradict the pre-set configuration:')
        for t in triggers: print('   -', t)
        print('  The extra seeds will NOT be trained and the test set will NOT be scored for this model.')
    else:
        print('  ' + ('Review decision recorded: KEEP_FINAL_CONFIGURATION. ' if triggers else 'No contradicting validation outcome. ') + 'Configuration frozen.')
    fz['models'][name] = {'status': status, 'weights': wc['chosen_weights'], 'threshold': tc['chosen_threshold'], 'tta': True,
        'rebalancing': False, 'augmentation': True, 'loss': 'composite ' + '+'.join(wc['components']),
        'validation_rows_seed42': {'baseline_CE': a1, 'B1_composite': b1, 'B2_threshold': b2, 'B3_tta': b3},
        'triggers': triggers, 'review_decision': REVIEW_DECISION if triggers else None,
        'frozen_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'), 'uses_test': False}
report_utils.save_result('final_config_frozen', fz)

**P10 - freeze gate**

Prints the validation rungs per model and then either `Configuration frozen` or `REVIEW REQUIRED` with the contradicting outcomes.

In [ ]:
# PART 2 - CELL P11: seeds 1337 and 2026
# v7 REPLACEMENT: each model trains with the loss its FROZEN configuration names.
# DeepLabV3+ frozen loss = agent-weighted composite; SegFormer-B3 frozen loss = plain CE (freeze-gate review 2026-11-14).
for seed in C.SEEDS[1:]:
    for name in PART2_MODELS:
        fzm = (report_utils.load_result('final_config_frozen') or {}).get('models', {}).get(name, {})
        if fzm.get('status') != 'FROZEN':
            print(f'[HOLD] {name}: seed {seed} NOT trained. The final configuration is not frozen yet (cell P10). No hours are spent on seeds before that.'); continue
        if fzm.get('loss', '').startswith('composite'):
            wc = report_utils.load_result(f'agent_weight_choice_{name}')
            if not wc:
                print(f'[wait] {name}: weight agent not finished (cell P5)'); continue
            loss_fn = composite_loss(wc['chosen_weights'])
        else:
            loss_fn = losses.build_loss('ce', C.NUM_CLASSES)
        if not os.path.exists(ckpt_final(f'{name}_final_seed{C.SEEDS[0]}')):
            print(f'[wait] {name}: seed-42 run first (cell P6)'); continue
        train_run(name, f'{name}_final_seed{seed}', loss_fn, seed=seed)
print('hours left:', round(hours_left(), 1))

**P11 - seeds 1337 and 2026**

Four more 6-epoch runs, only for a model whose configuration is frozen (otherwise `[HOLD]`). The longest cell. Resume-safe. A line starting `[stop]` means the session time guard ended it cleanly: follow step 8 of the run sequence.

In [ ]:
# PART 2 - CELL P12: test set
# v7 REPLACEMENT: threshold and TTA now come from each model's FROZEN configuration
# (TTA was hardcoded on; SegFormer-B3's frozen TTA choice comes from the measured candidates in cell P10).
# STEP 8 - TEST SET, final numbers. Needs all 3 seeds of a model AND the frozen final configuration from cell P10. Frozen choices only.
# Runs each (model, seed) once and remembers it; a finished model is never re-scored.
KEYS = ['miou', 'mean_dice', 'mean_precision', 'mean_recall', 'pixel_accuracy', 'boundary_f1']
for name in PART2_MODELS:
    out_name = f'test_final_{name}'
    done = report_utils.load_result(out_name)
    if done and done.get('complete'):
        print(f'[guard] {out_name} complete: test set is not scored again.'); continue
    fz = (report_utils.load_result('final_config_frozen') or {}).get('models', {}).get(name)
    if not fz or fz.get('status') != 'FROZEN':
        print(f'[HOLD] {name}: test NOT run. The final configuration is not frozen (see cell P11 output). Nothing was scored on test.'); continue
    thr = fz['threshold']; tta = fz.get('tta', True)
    paths = {s: ckpt_final(f'{name}_final_seed{s}') for s in C.SEEDS}
    missing = [s for s, p in paths.items() if not os.path.exists(p)]
    if missing:
        print(f'[wait] {name}: threshold frozen, seeds without weights={missing}. Finish cells P5-P11 first.'); continue
    per_seed = (done or {}).get('per_seed', [])
    test_loader = data.get_loader(frames['test'], 'test', batch_size=C.BATCH_SIZE[name])
    for s in C.SEEDS:
        if any(r['seed'] == s for r in per_seed):
            continue
        model = load_model(name, paths[s])
        m = engine.evaluate(model, test_loader, threshold=thr, tta=tta)
        m['seed'] = s
        per_seed.append(m)
        report_utils.save_result(out_name, {'model': name, 'per_seed': per_seed, 'complete': False,
                                            'threshold': thr, 'tta': tta, 'frozen_config_file': 'final_config_frozen.json'})
        print(f'{name} seed {s} TEST mIoU {m["miou"]:.4f}')
        del model; torch.cuda.empty_cache()
    agg = {}
    for k in KEYS:
        a = report_utils.mean_std_over_seeds(per_seed, k)
        a['std_sample_ddof1'] = float(np.std(a['values'], ddof=1)) if len(a['values']) > 1 else None
        agg[k] = a
    agg['per_class_iou_mean'] = np.nanmean([r['per_class_iou'] for r in per_seed], axis=0).tolist()
    report_utils.save_result(out_name, {'model': name, 'per_seed': per_seed, 'aggregate': agg, 'complete': True,
                                        'threshold': thr, 'tta': tta,
                                        'note': 'std = population std (ddof 0); std_sample_ddof1 is the sample std'})
    print(f'=== {name} TEST, {len(per_seed)} seeds (threshold {thr}, hflip TTA {tta}) ===')
    for k in KEYS:
        print(f'  {k:15s} {agg[k]["mean"]:.4f} +/- {agg[k]["std"]:.4f}  (values {[round(v, 4) for v in agg[k]["values"]]})')

**P12 - test set**

Only runs for a model whose configuration is FROZEN and has all 3 seeds. Prints each seed's test mIoU, then mean +/- std. Otherwise prints `[HOLD]` with the reason and scores nothing. A finished model is never scored again.

In [ ]:
# PART 2 - CELL P13: error analysis
# v7 REPLACEMENT: threshold and TTA come from each model's FROZEN configuration, and a saved
# error analysis is only reused if it matches the frozen choices (the earlier session's SegFormer-B3
# analysis used the composite weights and is recomputed with the frozen CE weights).
# STEP 9 - error analysis: confusion matrix, most confused classes, worst-case images
# Uses the seed-42 final model with its frozen threshold + TTA, on the VALIDATION set (no extra test-set exposure).
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import metrics as M
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def predict(model, x, thr, tta):
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(C.USE_AMP and dev == 'cuda')):
        probs = F.softmax(model(x).float(), dim=1)
        if tta:
            probs = (probs + torch.flip(F.softmax(model(torch.flip(x, dims=[3])).float(), dim=1), dims=[3])) / 2
    conf, pred = probs.max(dim=1)
    return torch.where(conf < thr, torch.zeros_like(pred), pred)

def show(ax, arr, title):
    ax.imshow(arr, cmap='tab20', vmin=0, vmax=12, interpolation='nearest'); ax.set_title(title, fontsize=8); ax.axis('off')

os.makedirs(os.path.join(C.FIGURES_DIR, 'failures'), exist_ok=True)
for name in PART2_MODELS:
    fzm = (report_utils.load_result('final_config_frozen') or {}).get('models', {}).get(name, {})
    thr = fzm.get('threshold'); tta = fzm.get('tta', True)
    path = ckpt_final(f'{name}_final_seed{C.SEEDS[0]}')
    ea = report_utils.load_result(f'error_analysis_{name}')
    if ea and ea.get('threshold') == thr and ea.get('tta') == tta and fzm.get('status') == 'FROZEN':
        print(f'[skip] error analysis for {name} already done with the frozen configuration'); continue
    if fzm.get('status') != 'FROZEN' or thr is None or not os.path.exists(path):
        print(f'[wait] {name}: needs the frozen configuration and the seed-42 final weights (cells P6-P10)'); continue
    model = load_model(name, path).to(dev).eval()
    vl = val_loader_for(name)
    cm = np.zeros((C.NUM_CLASSES, C.NUM_CLASSES), dtype=np.int64)
    img_scores = []
    for x, y in vl:
        pred = predict(model, x.to(dev), thr, tta).cpu().numpy(); y_np = y.numpy()
        for i in range(pred.shape[0]):
            c1 = M.confusion_matrix(pred[i], y_np[i], C.NUM_CLASSES)
            cm += c1
            s = M.metrics_from_confusion(c1)['miou']
            img_scores.append(1.0 if np.isnan(s) else float(s))
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(C.NUM_CLASSES), C.CLASS_NAMES, rotation=90); ax.set_yticks(range(C.NUM_CLASSES), C.CLASS_NAMES)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'Pixel confusion matrix, row-normalised (validation, {name}, seed 42)')
    fig.tight_layout(); fig.savefig(os.path.join(C.FIGURES_DIR, f'confusion_matrix_{name}.png'), dpi=140); plt.close(fig)
    pairs = sorted(((float(cmn[i, j]), C.CLASS_NAMES[i], C.CLASS_NAMES[j]) for i in range(C.NUM_CLASSES)
                    for j in range(C.NUM_CLASSES) if i != j and cmn[i, j] > 0), reverse=True)
    pairs_nobg = [p for p in pairs if 'background' not in (p[1], p[2])]
    per_class = M.metrics_from_confusion(cm)
    # worst 8 validation images (the loader does not shuffle, so position = index in the dataset)
    worst = np.argsort(img_scores)[:8]
    fig, axes = plt.subplots(len(worst), 3, figsize=(7, 2.3 * len(worst)))
    for r, idx in enumerate(worst):
        x, y = vl.dataset[int(idx)]
        p = predict(model, x.unsqueeze(0).to(dev), thr, tta)[0].cpu().numpy()
        img = (x.permute(1, 2, 0).numpy() * np.array(C.IMAGENET_STD) + np.array(C.IMAGENET_MEAN)).clip(0, 1)
        axes[r, 0].imshow(img); axes[r, 0].set_title(f'val image {int(idx)} (image mIoU {img_scores[idx]:.2f})', fontsize=8); axes[r, 0].axis('off')
        show(axes[r, 1], y.numpy(), 'ground truth'); show(axes[r, 2], p, 'prediction')
    fig.tight_layout(); fig.savefig(os.path.join(C.FIGURES_DIR, 'failures', f'worst8_{name}.png'), dpi=120); plt.close(fig)
    report_utils.save_result(f'error_analysis_{name}', {
        'model': name, 'split': 'validation', 'seed': C.SEEDS[0], 'threshold': thr, 'tta': tta,
        'confusion_matrix_counts': cm.tolist(), 'confusion_matrix_row_normalised': cmn.tolist(),
        'top_confused_pairs_all': pairs[:10], 'top_confused_pairs_excluding_background': pairs_nobg[:10],
        'per_class_iou_val': dict(zip(C.CLASS_NAMES, per_class['per_class_iou'])),
        'worst_val_image_indices': [int(i) for i in worst], 'worst_val_image_miou': [img_scores[i] for i in worst]})
    print(f'--- {name}: most confused class pairs (true -> predicted, share of the true class pixels) ---')
    for frac, a, b in pairs[:6]: print(f'  {a:12s} -> {b:12s} {frac:.1%}')
    print('  excluding background:')
    for frac, a, b in pairs_nobg[:6]: print(f'  {a:12s} -> {b:12s} {frac:.1%}')
    print('  per-class IoU:', {k: (None if v != v else round(v, 3)) for k, v in zip(C.CLASS_NAMES, per_class['per_class_iou'])})
    del model; torch.cuda.empty_cache()

**P13 - error analysis**

Prints the most confused class pairs and per-class IoU and saves the confusion matrix and the worst 8 validation images.

In [ ]:
# PART 2 - CELL P14: completeness check and download
# completeness check, headline numbers, and the zip of every new result, log and figure
import zipfile
REQ = []
for name in PART2_MODELS:
    t = report_utils.load_result(f'test_final_{name}')
    REQ += [(f'{name}: weight agent (trial log + choice)', bool(report_utils.load_result(f'agent_weight_choice_{name}')) and os.path.exists(os.path.join(C.RESULTS_DIR, f'agent_weight_trials_{name}.csv'))),
            (f'{name}: threshold agent (trial log + choice)', bool(report_utils.load_result(f'agent_threshold_choice_{name}')) and os.path.exists(os.path.join(C.RESULTS_DIR, f'agent_threshold_trials_{name}.csv'))),
            (f'{name}: 3 seeds of the final configuration trained', all(os.path.exists(ckpt_final(f'{name}_final_seed{s}')) for s in C.SEEDS)),
            (f'{name}: final configuration frozen after the validation comparisons (cell P10)', ((report_utils.load_result('final_config_frozen') or {}).get('models', {}).get(name, {}).get('status') == 'FROZEN')),
            (f'{name}: test result, mean +/- std over 3 seeds', bool(t and t.get('complete'))),
            (f'{name}: no-augmentation ablation', bool(aug_off_val(name))),
            (f'{name}: ablation tables (one at a time + cumulative)', bool(report_utils.load_result(f'ablation_ladder_{name}'))),
            (f'{name}: error analysis + confusion matrix + failure images', bool(report_utils.load_result(f'error_analysis_{name}')))]
print('REQUIRED ITEMS')
for k, ok in REQ: print(f'  [{"x" if ok else " "}] {k}')
complete = all(ok for _, ok in REQ)
report_utils.save_result('part2_status', {'complete': complete, 'items': {k: ok for k, ok in REQ}, 'time': time.strftime('%Y-%m-%d %H:%M:%S %Z')})
if complete:
    print('\nALL REQUIRED PART 2 ITEMS ARE COMPLETE.')
    for name in PART2_MODELS:
        a = report_utils.load_result(f'test_final_{name}')['aggregate']['miou']
        print(f'{name} TEST mIoU over {a["n_seeds"]} seeds: {a["mean"]:.4f} +/- {a["std"]:.4f}')
else:
    print('\n' + '!' * 70 + '\nINCOMPLETE: the items marked [ ] above are NOT done yet. This is not a finished result.\n'
          'Save Version, then start a new session with this version\'s Output attached (and coco-seg-project) and run again.\n' + '!' * 70)
bundle = '/kaggle/working/part2_results_bundle.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for pat in ['*.json', '*.csv', '*.db', 'figures/*.png', 'figures/failures/*.png']:
        for p in glob.glob(os.path.join(C.RESULTS_DIR, pat)):
            z.write(p, os.path.relpath(p, C.PROJECT_DIR))
print('Saved', bundle, round(os.path.getsize(bundle) / 1e6, 1), 'MB;', len(zipfile.ZipFile(bundle).namelist()), 'files')


**P14 - completeness check and download**

Lists every required item with [x] or [ ], then `ALL REQUIRED PART 2 ITEMS ARE COMPLETE` or an INCOMPLETE banner, then `Saved /kaggle/working/part2_results_bundle.zip`.